(net-selectivity-notebook)=
# Correct proportions for net selectivity


In [1]:
%run ./number_proportions.ipynb

C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:717: UserWarning: Override: Using input 'ship_id' instead of 'ship' from 'biodata.catch['US']' DataFrame input.
  warnings.warn(
C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:717: UserWarning: Override: Using input 'ship_id' instead of 'ship' from 'biodata.catch['CAN']' DataFrame input.
  warnings.warn(
C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:717: UserWarning: Override: Using input 'survey_id' instead of 'survey' from 'biodata.catch['US']' DataFrame input.
  warnings.warn(
C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:717: UserWarning: Override: Using input 'survey_id' instead of 'survey' from 'biodata.catch['CAN']' DataFrame input.
  warnings.warn(
C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:732: UserWarning: Override: Using input 'species_id' instead of 'species_code' from 'biodata.catch' DataFrame input.
  warnings.warn(
C:\Users\Brandyn\GitHub\echopop\echopop\utils.py:717: UserWarning: Override: Using input 'ship_

<xarray.Dataset> Size: 200kB
Dimensions:     (stratum_ks: 9, length_bin: 40, age_bin: 22, sex: 3)
Coordinates:
  * stratum_ks  (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin  (length_bin) category 680B (1.0, 3.0] ... (79.0, 81.0]
  * age_bin     (age_bin) category 374B (0.5, 1.5] (1.5, 2.5] ... (21.5, 22.5]
  * sex         (sex) object 24B 'female' 'male' 'unsexed'
Data variables:
    aged        (stratum_ks, length_bin, age_bin, sex) int64 190kB 0 0 0 ... 0 0
    unaged      (stratum_ks, length_bin, sex) float64 9kB nan nan ... 0.0 0.0

<xarray.Dataset> Size: 381kB
Dimensions:             (stratum_ks: 9, length_bin: 40, age_bin: 22, sex: 2)
Coordinates:
  * stratum_ks          (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin          (length_bin) category 680B (1.0, 3.0] ... (79.0, 81.0]
  * age_bin             (age_bin) category 374B (0.5, 1.5] ... (21.5, 22.5]
  * sex                 (sex) object 16B 'female' 'male'
Data variables:
    count               (stratum_ks, length_bin, age_bin, sex) int64 127kB 0 ...
    proportion          (stratum_ks, length_bin, age_bin, sex) float64 127kB ...
    proportion_overall  (stratum_ks, length_bin, age_bin, sex) float64 127kB ...

## Import necessary modules

With the number proportions calculated, we can optionally correct them for length-based net selectivity. This is done via the `selectivity` module of the `survey` sub-package.

In [2]:
from echopop.survey import selectivity, proportions

Net selectivity diverges from the standard workflow at the count-binning stage. Selectivity expansion is applied *before* tabulating counts and number proportions, and weight proportions are then derived from those corrected number proportions using fitted length-binned weights.

![Net selectivity workflow steps](../_static/net_selectivity_workflow_steps.svg)

## Assigning selectivity expansion factors

In the selectivity pathway, corrections are applied at the specimen level before any stratum-level normalization. If $S(L_i)$ is the retention probability for specimen $i$ at length $L_i$, then the effective selectivity [expansion factor can be assigned to each animal](../theory/selectivity.md#specimen-level-expansion). The key function in this step is {func}`assign_selectivity_expansion <echopop.survey.selectivity.assign_selectivity_expansion>`.

Core arguments are:

- `biodata`: Specimen-level `pandas.DataFrame` containing at least length and net-type columns.
- `net_selectivity_params`: Selectivity parameters in one of two accepted formats.
- `net_column`: Name of the net-type column used to map parameters (for example, `"gear"`).
- `minimum_selectivity`: Numerical floor $S_{\min}$ used to stabilize inverse weighting.

`net_selectivity_params` supports **both** of the following structures:

1. **Single global parameter set** (applies the same ogive to all nets):

```python
{"l50": 10.9, "sr": 14.0}
```

2. **Net-specific parameter dictionary** (different parameters by net/gear type):

```python
{
    "AWT": {"l50": 10.9, "sr": 14.0},
    "IKMT": {"intercept": -8.2, "slope": 0.58},
}
```

Each parameter block must be either (`l50`, `sr`) **or** (`intercept`, `slope`). This allows mixed parameterization across net types while still mapping to the same logistic selectivity formulation.

In [3]:
# Option A: single global selectivity ogive (used for all nets)
NET_SELECTIVITY_GLOBAL = {
    "l50": 10.9,
    "sr": 14.0,
}

# Option B: net-specific selectivity parameters (different ogives by gear)
# Keys should match values in specimen_data["gear"]
NET_SELECTIVITY_BY_NET = {
    "AWT": {"l50": 10.9, "sr": 14.0},
    "IKMT": {"intercept": -8.2, "slope": 0.58},
}

### Preparing the data

In [4]:
# Create dummy copy for demonstration
import copy

dict_df_bio["catch"]["gear"] = "AWT"

For this example, we will only consider the specimen-specific data since that is the intended data format.

In [5]:
specimen_data = dict_df_bio["specimen"]

We can inspect the catch data to see the various (or singular) net-types used during the survey.

In [6]:
dict_df_bio["catch"].gear.unique()

array(['AWT'], dtype=object)

However, `specimen_data` does not contain this column:

In [7]:
"gear" in specimen_data

False

Since the `"gear"` column does not exist in `specimen_data`, we have to merge that information from `dict_df_bio["catch"]`:

In [8]:
specimen_data = dict_df_bio["specimen"].merge(
    dict_df_bio["catch"][["uid", "gear"]].drop_duplicates("uid"),
    on="uid",
    how="left",
)

We can now see that `specimen_data` contains this column:

In [9]:
"gear" in specimen_data

True

In [10]:
specimen_data["gear"].unique()

array(['AWT'], dtype=object)

### Assigning expansion factors to each row

Since only one net-type is present in `specimen_data`, the `NET_SELECTIVITY_GLOBAL` dictionary can be used since it will uniformly apply those parameters across all rows:

In [11]:
# Apply selectivity expansion at the specimen level using the selected NET_SELECTIVITY definition
specimen_data_selectivity = selectivity.assign_selectivity_expansion(
    specimen_data,
    NET_SELECTIVITY_GLOBAL,
    net_column="gear",
    minimum_selectivity=1e-12,
)


The output `pandas.DataFrame` now has three additional columns: `l50`, `sr`, and `selectivity_expansion`

In [12]:
specimen_data_selectivity.filter(["sex", "length", "uid", "gear", "l50", "sr", "selectivity_expansion"])

,sex,length,uid,gear,l50,sr,selectivity_expansion
0,male,24.0,2-2-99999-1.0,AWT,10.9,14.0,1.067928
1,female,23.0,2-2-99999-1.0,AWT,10.9,14.0,1.184647
2,female,22.0,2-2-99999-1.0,AWT,10.9,14.0,1.501923
3,male,22.0,2-2-99999-1.0,AWT,10.9,14.0,1.501923
4,female,23.0,2-2-99999-1.0,AWT,10.9,14.0,1.184647
...,...,...,...,...,...,...,...
3260,female,46.0,2-2-99999-222.0,AWT,10.9,14.0,1.000000
3261,female,46.0,2-2-99999-222.0,AWT,10.9,14.0,1.000000
3262,female,52.0,2-2-99999-222.0,AWT,10.9,14.0,1.000000
3263,female,48.0,2-2-99999-222.0,AWT,10.9,14.0,1.000000


## Converting to corrected number proportions

These selectivity expansion factors are first summed across all indices *including* the haul-level identifier `uid`. This is to maintain good habits since there may be cases where a dataset has multiple net-types that will map different selectivity parameters to specific haul numbers.

In [13]:
# Effective counts per haul (length × age × sex × stratum × uid)
da_count_distribution_hauls = proportions.compute_binned_counts(
    data=specimen_data_selectivity.dropna(subset=["length"]),
    groupby_cols=["stratum_ks", "length_bin", "age_bin", "sex", "uid"],
    count_col="selectivity_expansion",
    agg_func="sum",
)

These can then be summed over each stratum:

In [14]:
# Collapse to stratum level
da_count_distribution_strata = da_count_distribution_hauls.sum(dim=["uid"])

The last step is then to normalize these effective counts by converting them into proportions:

In [15]:
ds_number_proportions_corrected = proportions.number_proportions(
    data=da_count_distribution_strata,
    group_columns=["stratum_ks"],
    exclude_filters={"sex": "unsexed"},
)

In [16]:
ds_number_proportions_corrected

<xarray.Dataset> Size: 381kB
Dimensions:             (stratum_ks: 9, length_bin: 40, age_bin: 22, sex: 2)
Coordinates:
  * stratum_ks          (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin          (length_bin) category 680B (1.0, 3.0] ... (79.0, 81.0]
  * age_bin             (age_bin) category 374B (0.5, 1.5] ... (21.5, 22.5]
  * sex                 (sex) object 16B 'female' 'male'
Data variables:
    count               (stratum_ks, length_bin, age_bin, sex) float64 127kB ...
    proportion          (stratum_ks, length_bin, age_bin, sex) float64 127kB ...
    proportion_overall  (stratum_ks, length_bin, age_bin, sex) float64 127kB ...

The corrected output is an `xarray.Dataset` returned by `proportions.number_proportions`. It retains the original dimensional structure (typically stratum × length_bin × age_bin × sex), but with selectivity-adjusted effective counts embedded in the normalization. In practice, two variables are central: `proportion` (within-group normalized proportions) and `proportion_overall` (overall normalized proportions for downstream weighting and slicing operations). If age is absent in the source specimen records, the output naturally reduces to a length-based structure. The differences in the selectivity-corrected and uncorrected number proportions for a few example strata show how the adjustment varies depending on the length range a stratum comprises.

In [17]:
# ORIGINAL NON-CORRECTED WORKFLOW
da_count_orig = proportions.compute_binned_counts(
    data=specimen_data.dropna(subset=["length"]),
    groupby_cols=["stratum_ks", "length_bin", "age_bin", "sex", "uid"],
    count_col="length",
    agg_func="size",
)

ds_number_proportions_orig = proportions.number_proportions(
    data=da_count_orig,
    group_columns=["stratum_ks"]
)


In [18]:
import xarray as xr
import holoviews as hv
import pandas as pd
import numpy as np
from typing import Any, Union
from echopop.graphics.diagnostics import Diagnostics

def plot_selectivity_distribution_comparison(
    self,
    uncorrected: Union[xr.Dataset, xr.DataArray],
    corrected: Union[xr.Dataset, xr.DataArray],
    variable: str = "proportion_overall",
    stratum_dim: str = "stratum_ks",
    length_dim: str = "length_bin",
    ncols: int = 3,
    panel_width: int = 360,
    panel_height: int = 240,
) -> hv.Layout:
    """
    Interactively compare uncorrected and selectivity-corrected length distributions by stratum.

    Parameters
    ----------
    uncorrected : xarray.Dataset or xarray.DataArray
        Distribution output without net-selectivity correction.
    corrected : xarray.Dataset or xarray.DataArray
        Distribution output after net-selectivity correction.
    variable : str, default "proportion_overall"
        Variable to plot when inputs are Datasets. If unavailable, ``"proportion"`` is used as
        a fallback.
    stratum_dim : str, default "stratum_ks"
        Name of the stratum dimension used for interactive grouping.
    length_dim : str, default "length_bin"
        Name of the length-bin dimension used as the x-axis distribution coordinate.
    ncols : int, default 3
        Number of columns used when laying out all stratum panels.
    panel_width : int, default 360
        Width of each stratum panel.
    panel_height : int, default 240
        Height of each stratum panel.

    Returns
    -------
    holoviews.Layout
        Multi-panel layout with one panel per stratum and overlaid uncorrected vs corrected
        distributions using numeric length-bin x-axis values. A single legend is shown in the
        first panel only.
    """

    def _extract_distribution_array(data: Union[xr.Dataset, xr.DataArray]) -> xr.DataArray:
        if isinstance(data, xr.DataArray):
            return data
        if not isinstance(data, xr.Dataset):
            raise TypeError("Inputs must be xarray.Dataset or xarray.DataArray.")

        if variable in data:
            return data[variable]
        if "proportion" in data:
            return data["proportion"]

        raise KeyError(
            f"Variable '{variable}' not found and fallback 'proportion' is unavailable."
        )

    def _prepare_frame(data: Union[xr.Dataset, xr.DataArray], label: str) -> pd.DataFrame:
        distribution = _extract_distribution_array(data)

        if stratum_dim not in distribution.dims:
            raise KeyError(
                f"Stratum dimension '{stratum_dim}' was not found in: {distribution.dims}."
            )
        if length_dim not in distribution.dims:
            raise KeyError(
                f"Length dimension '{length_dim}' was not found in: {distribution.dims}."
            )

        # Collapse all non-target dimensions into the plotted stratum-length distribution
        reduce_dims = [dim for dim in distribution.dims if dim not in [stratum_dim, length_dim]]
        if reduce_dims:
            distribution = distribution.sum(dim=reduce_dims)

        frame = distribution.to_series().rename("value").reset_index()
        frame["source"] = label
        return frame[[stratum_dim, length_dim, "value", "source"]]

    def _length_to_numeric(value: Any) -> float:
        if pd.isna(value):
            return np.nan

        if isinstance(value, pd.Interval):
            return float(value.mid)

        if hasattr(value, "mid"):
            return float(value.mid)

        try:
            return float(value)
        except (TypeError, ValueError):
            text = str(value).strip()
            cleaned = text.translate(str.maketrans("", "", "[]()"))
            parts = [part.strip() for part in cleaned.split(",") if part.strip() != ""]
            if len(parts) == 2:
                try:
                    return (float(parts[0]) + float(parts[1])) / 2.0
                except ValueError:
                    return np.nan
            return np.nan

    def _stratum_sort_key(value: Any) -> tuple:
        try:
            return (0, float(value))
        except (TypeError, ValueError):
            text = str(value).strip()
            try:
                return (0, float(text))
            except ValueError:
                return (1, text)

    uncorrected_frame = _prepare_frame(uncorrected, "Uncorrected")
    corrected_frame = _prepare_frame(corrected, "Selectivity-corrected")
    plot_data = pd.concat([uncorrected_frame, corrected_frame], ignore_index=True)

    # Always map length bins to a numeric axis (e.g., bin midpoints for interval bins)
    plot_data["length"] = plot_data[length_dim].map(_length_to_numeric)
    plot_data = plot_data.dropna(subset=["length", "value"]).sort_values("length")

    strata = sorted(
        pd.Index(plot_data[stratum_dim].dropna().drop_duplicates().tolist()),
        key=_stratum_sort_key,
    )
    panels = []
    for panel_index, stratum in enumerate(strata):
        panel_data = plot_data[plot_data[stratum_dim] == stratum]

        uncorrected_panel = panel_data[panel_data["source"] == "Uncorrected"]
        corrected_panel = panel_data[panel_data["source"] == "Selectivity-corrected"]

        uncorrected_curve = hv.Curve(
            uncorrected_panel,
            kdims=["length"],
            vdims=["value"],
            label="Uncorrected",
        ).opts(color="#3b82f6", line_width=2.5, alpha=0.95)

        corrected_curve = hv.Curve(
            corrected_panel,
            kdims=["length"],
            vdims=["value"],
            label="Selectivity-corrected",
        ).opts(color="#ef4444", line_width=2.5, alpha=0.95)

        panel = (uncorrected_curve * corrected_curve).opts(
            show_legend=panel_index == 0,
            legend_position="top_right",
            xlabel="Length",
            ylabel="Distribution",
            title=f"Stratum: {stratum}",
            width=panel_width,
            height=panel_height,
        )
        panels.append(panel)

    if not panels:
        raise ValueError("No strata were available to plot.")

    return hv.Layout(panels).cols(ncols)

Diagnostics.plot_selectivity_distribution_comparison = plot_selectivity_distribution_comparison
# Initialize the plotting class
dplot = Diagnostics()

In [30]:
import holoviews as hv
plot = dplot.plot_selectivity_distribution_comparison(
    ds_number_proportions_orig.sel(stratum_ks=[1,2,7]), 
    ds_number_proportions_corrected.sel(stratum_ks=[1,2,7]),
    ncols=1,
    panel_width = 460,
    panel_height = 240,
)
hv.save(plot, '../_static/net_selectivity.html', backend='bokeh')

<iframe src="../_static/net_selectivity.html" width="500" height="760"></iframe>

## Converting to corrected weight proportions

After corrected number proportions are obtained, weight proportions are computed by combining those corrected distributions with fitted mean length-binned weights. In this workflow (combined aged/unaged pathway), Echopop uses `proportions.fitted_weight_proportions_combined`.

Conceptually, corrected number proportions define the conditional composition across age/sex within each length bin, while fitted length weights provide the length-marginal mass scaling. The resulting dataset is re-normalized within each stratum and remains dimensionally compatible with downstream abundance and biomass apportionment.

The key function in this step is `proportions.fitted_weight_proportions_combined`, with core arguments:

- `number_proportions`: Corrected number-proportion `xarray.Dataset` containing `proportion` and `proportion_overall`.
- `binned_weights`: Fitted length-binned weights aligned on `length_bin` (typically `sex="all"` weights).
- `stratum_dim`: Stratum dimensions that define normalization groups (e.g., `["stratum_ks"]`).

In [21]:
# Derive weight proportions from selectivity-corrected number proportions using fitted length-binned weights
ds_weight_proportions_corrected = proportions.fitted_weight_proportions_combined(
    number_proportions=ds_number_proportions_corrected,
    binned_weights=da_binned_weights_all,
    stratum_dim=["stratum_ks"],
)

In [22]:
ds_weight_proportions_corrected

<xarray.Dataset> Size: 1kB
Dimensions:             (stratum_ks: 9, length_bin: 40, age_bin: 22, sex: 0)
Coordinates:
  * stratum_ks          (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin          (length_bin) category 680B (1.0, 3.0] ... (79.0, 81.0]
  * age_bin             (age_bin) category 374B (0.5, 1.5] ... (21.5, 22.5]
  * sex                 (sex) object 0B 
Data variables:
    proportion_overall  (stratum_ks, length_bin, age_bin, sex) float64 0B

The output is an `xarray.Dataset` with `proportion_overall`, representing weight-scaled proportions that sum to 1 within each stratum under the provided `stratum_dim` definition. Because the calculation is driven by corrected number proportions, the age/sex/length structure remains coherent with the corrected count pathway.

## Integrating into the general workflow

With `ds_number_proportions_corrected` and `ds_weight_proportions_corrected` computed, the net-selectivity pathway can be dropped into the same downstream steps used by the standard FEAT workflow: abundance apportionment, biomass apportionment, age filters, and geostatistical projection. The only structural difference is where correction enters the pipeline (specimen-level expansion before count tabulation).